# V3 Colab setup and freeze guard
Mount Drive, clone the pinned repository, copy the user-uploaded frozen V3 package to local Colab storage, and validate every checksum before any training.

In [ ]:
from pathlib import Path
import importlib.metadata as package_metadata
import os, shutil, stat, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/kltn')
PINNED_COMMIT = 'f39b35d'  # update only when deliberately locking a new code commit
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/maiphuowng205/kltn.git', str(REPO)], check=True)
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', PINNED_COMMIT], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', PINNED_COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements-colab.txt')], check=True)
V3_DRIVE_ROOT = Path('/content/drive/MyDrive/kltn/frozen/vn_v3_lseg_2026-08-03')
DRIVE_RUN_ROOT = Path('/content/drive/MyDrive/kltn/runs')
WORKSPACE = Path('/content/vn_v3_workspace')
DATA_ROOT = WORKSPACE / 'data' / 'lseg_v3'
if not (V3_DRIVE_ROOT / 'data' / 'lseg_v3').exists() and not (V3_DRIVE_ROOT / 'curated').exists():
    raise FileNotFoundError('Upload the frozen V3 package to V3_DRIVE_ROOT before continuing.')
if (V3_DRIVE_ROOT / 'data' / 'lseg_v3').exists():
    if (WORKSPACE / 'data').exists(): shutil.rmtree(WORKSPACE / 'data')
    shutil.copytree(V3_DRIVE_ROOT / 'data', WORKSPACE / 'data')
else:
    if DATA_ROOT.exists(): shutil.rmtree(DATA_ROOT)
    DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(V3_DRIVE_ROOT, DATA_ROOT)
# Freeze data is an input, never a run-output location.  Remove write bits
# from the local copy so accidental writes fail loudly during the run.
for path in DATA_ROOT.rglob('*'):
    mode = path.stat().st_mode
    path.chmod(mode & ~stat.S_IWUSR & ~stat.S_IWGRP & ~stat.S_IWOTH)
WORKSPACE.joinpath('runs').mkdir(parents=True, exist_ok=True)
commit = subprocess.check_output(['git','rev-parse','HEAD'], cwd=REPO, text=True).strip()
def version(name):
    try: return package_metadata.version(name)
    except package_metadata.PackageNotFoundError: return None
runtime = {'python': sys.version, 'git_commit': commit, 'device': 'cuda' if __import__('torch').cuda.is_available() else 'cpu', 'packages': {name: version(name) for name in ['numpy','pandas','pyarrow','scikit-learn','cvxpy','torch','xgboost']}}
(WORKSPACE / 'runs' / 'setup_environment.json').write_text(__import__('json').dumps(runtime, indent=2), encoding='utf-8')
DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copy2(WORKSPACE / 'runs' / 'setup_environment.json', DRIVE_RUN_ROOT / 'setup_environment.json')
print({'repo': str(REPO), 'commit': commit, 'data_root': str(DATA_ROOT), 'runtime': runtime})


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(REPO / 'scripts' / 'validate_v3_contract.py'), '--workspace-root', str(WORKSPACE)], check=True)
